# 02 — Tratamento (heurístico)

Testa transformações **clássicas** (sem rede neural) sobre as fotos de campo. Cada seção mostra o antes e o depois nas **mesmas** fotos.
Nenhum parâmetro aqui é decisão final: são valores de partida pra gente olhar e escolher junto.

1. Nitidez (foto borrada?)
2. Iluminação e contraste
3. Kernels de convolução (cada matriz escrita à mão)
4. Geometria (tamanho, proporção, rotação)

In [ ]:
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA = Path(os.environ.get("LEITURISTA_DATA", Path.home() / "Developer/leiturista/data"))

# Todas as fotos no disco (aqui não usamos o CSV, então as órfãs também servem).
todas = sorted(DATA.glob("distribuidora_campo/*/*.jpg"))
print(len(todas), "fotos")

# 8 fotos fixas (seed 0) usadas em todas as seções, pra comparar justo.
rng = np.random.default_rng(0)
exemplos = [todas[i] for i in rng.choice(len(todas), 8, replace=False)]


def ler_cinza(caminho):
    return cv2.imread(str(caminho), cv2.IMREAD_GRAYSCALE)


def ler_cor(caminho):
    # OpenCV lê em BGR; matplotlib espera RGB.
    return cv2.cvtColor(cv2.imread(str(caminho)), cv2.COLOR_BGR2RGB)


def mostrar_linha(imagens, titulos, tamanho=2.4):
    fig, eixos = plt.subplots(1, len(imagens), figsize=(len(imagens) * tamanho, tamanho * 1.4))
    for eixo, imagem, titulo in zip(np.ravel(eixos), imagens, titulos):
        eixo.imshow(imagem, cmap="gray" if imagem.ndim == 2 else None, vmin=0, vmax=255)
        eixo.set_title(titulo, fontsize=8)
        eixo.axis("off")
    plt.tight_layout()
    plt.show()

## 1. Nitidez

Ideia clássica: foto nítida tem bordas fortes; foto borrada tem bordas fracas. Dois scores:

- **Variância do Laplaciano**: o Laplaciano realça mudanças bruscas de brilho; a variância do resultado é alta quando há muita borda.
- **Energia do gradiente (Sobel)**: média da magnitude do gradiente horizontal e vertical.

In [ ]:
def score_laplaciano(cinza):
    laplaciano = cv2.Laplacian(cinza, cv2.CV_64F)
    return laplaciano.var()


def score_sobel(cinza):
    gradiente_x = cv2.Sobel(cinza, cv2.CV_64F, 1, 0, ksize=3)
    gradiente_y = cv2.Sobel(cinza, cv2.CV_64F, 0, 1, ksize=3)
    return np.sqrt(gradiente_x ** 2 + gradiente_y ** 2).mean()


tabela = pd.DataFrame({
    "foto": [c.name[-14:] for c in exemplos],
    "laplaciano": [score_laplaciano(ler_cinza(c)) for c in exemplos],
    "sobel": [score_sobel(ler_cinza(c)) for c in exemplos],
})
tabela.round(1)

In [ ]:
# As 8 fotos ordenadas da mais borrada pra mais nítida (pelo Laplaciano).
ordem = tabela.laplaciano.argsort().tolist()
mostrar_linha(
    [ler_cor(exemplos[i]) for i in ordem],
    [f"lap={tabela.laplaciano[i]:.0f}" for i in ordem],
)

**Pra olhar:** a ordem dos scores bate com o que você enxerga como borrado? Uma foto escura ou de parede lisa também tem score baixo sem estar desfocada: o score mede "tem borda", não "está em foco".

## 2. Iluminação e contraste

Três técnicas em tons de cinza:

- **Equalização de histograma global**: espalha os níveis de brilho pelo intervalo todo. Simples, mas pode estourar regiões já claras.
- **CLAHE**: equaliza por blocos pequenos com limite de contraste; costuma preservar melhor detalhe local.
- **Correção gama**: `saída = 255 * (entrada/255) ** gama`. gama < 1 clareia, gama > 1 escurece.

In [ ]:
def equalizar_global(cinza):
    return cv2.equalizeHist(cinza)


def clahe(cinza, limite=2.0, bloco=8):
    return cv2.createCLAHE(clipLimit=limite, tileGridSize=(bloco, bloco)).apply(cinza)


def corrigir_gama(cinza, gama):
    tabela_de_busca = np.array([255 * (n / 255) ** gama for n in range(256)], dtype=np.uint8)
    return cv2.LUT(cinza, tabela_de_busca)


for caminho in exemplos[:4]:
    cinza = ler_cinza(caminho)
    mostrar_linha(
        [cinza, equalizar_global(cinza), clahe(cinza), corrigir_gama(cinza, 0.5), corrigir_gama(cinza, 1.8)],
        ["original", "equalização global", "CLAHE (limite 2, bloco 8)", "gama 0.5 (clareia)", "gama 1.8 (escurece)"],
    )

## 3. Kernels de convolução

Um kernel é uma matriz pequena que desliza sobre a imagem; em cada posição multiplica-se elemento a elemento e soma-se o resultado.
Abaixo, **cada kernel está escrito como matriz** e aplicado com `cv2.filter2D` (é a convolução; o OpenCV só faz a conta rápido).

In [ ]:
# Suavização: média ponderada da vizinhança. Reduz ruído, borra bordas.
KERNEL_MEDIA_3x3 = np.ones((3, 3), dtype=np.float32) / 9

KERNEL_GAUSSIANO_3x3 = np.array([
    [1, 2, 1],
    [2, 4, 2],
    [1, 2, 1],
], dtype=np.float32) / 16   # centro pesa mais que as pontas

# Bordas: diferença entre lados opostos.
KERNEL_SOBEL_X = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1],
], dtype=np.float32)        # responde a bordas VERTICAIS (mudança na horizontal)

KERNEL_SOBEL_Y = KERNEL_SOBEL_X.T   # responde a bordas HORIZONTAIS

KERNEL_LAPLACIANO = np.array([
    [0,  1, 0],
    [1, -4, 1],
    [0,  1, 0],
], dtype=np.float32)        # borda em qualquer direção (segunda derivada)

# Realce de nitidez: imagem original + bordas (soma dos pesos = 1, não muda o brilho médio).
KERNEL_NITIDEZ = np.array([
    [ 0, -1,  0],
    [-1,  5, -1],
    [ 0, -1,  0],
], dtype=np.float32)

KERNELS = {
    "média 3x3": KERNEL_MEDIA_3x3,
    "gaussiano 3x3": KERNEL_GAUSSIANO_3x3,
    "Sobel X": KERNEL_SOBEL_X,
    "Sobel Y": KERNEL_SOBEL_Y,
    "Laplaciano": KERNEL_LAPLACIANO,
    "nitidez": KERNEL_NITIDEZ,
}

for nome, kernel in KERNELS.items():
    print(f"{nome}  (soma = {kernel.sum():.2f})\n{kernel}\n")

In [ ]:
def aplicar(cinza, kernel):
    # -1 = mesma profundidade da entrada; valores negativos são cortados em 0.
    return cv2.filter2D(cinza, -1, kernel)


def magnitude_bordas(cinza):
    # Em float pra não perder o sinal (borda escuro->claro é negativa em um dos sentidos).
    bx = cv2.filter2D(cinza.astype(np.float32), -1, KERNEL_SOBEL_X)
    by = cv2.filter2D(cinza.astype(np.float32), -1, KERNEL_SOBEL_Y)
    return np.clip(np.sqrt(bx ** 2 + by ** 2), 0, 255).astype(np.uint8)


for caminho in exemplos[:3]:
    cinza = ler_cinza(caminho)
    mostrar_linha(
        [cinza, aplicar(cinza, KERNEL_MEDIA_3x3), aplicar(cinza, KERNEL_GAUSSIANO_3x3),
         magnitude_bordas(cinza), np.clip(aplicar(cinza, KERNEL_LAPLACIANO) * 8, 0, 255), aplicar(cinza, KERNEL_NITIDEZ)],
        ["original", "média", "gaussiano", "Sobel (magnitude)", "Laplaciano (x8 pra ver)", "nitidez"],
    )

### Filtro não linear: mediana

Não é convolução (substitui o pixel pela **mediana** da vizinhança). Remove ruído do tipo "sal e pimenta" borrando menos as bordas que a média.

In [ ]:
for caminho in exemplos[:2]:
    cinza = ler_cinza(caminho)
    mostrar_linha(
        [cinza, cv2.medianBlur(cinza, 3), cv2.medianBlur(cinza, 5), aplicar(cinza, KERNEL_MEDIA_3x3)],
        ["original", "mediana 3", "mediana 5", "média 3x3 (comparação)"],
    )

## 4. Geometria

### 4.1 Tamanho e proporção
As fotos são 360x480 (proporção 3:4). Uma entrada quadrada pra CNN exige escolher: **esticar** (distorce), **cortar** (perde borda) ou **letterbox** (preserva tudo, adiciona faixas).

In [ ]:
def letterbox(imagem, lado, cor_da_faixa=0):
    altura, largura = imagem.shape[:2]
    escala = lado / max(altura, largura)
    nova_largura, nova_altura = round(largura * escala), round(altura * escala)
    redimensionada = cv2.resize(imagem, (nova_largura, nova_altura), interpolation=cv2.INTER_AREA)

    quadro = np.full((lado, lado) + imagem.shape[2:], cor_da_faixa, dtype=imagem.dtype)
    topo = (lado - nova_altura) // 2
    esquerda = (lado - nova_largura) // 2
    quadro[topo:topo + nova_altura, esquerda:esquerda + nova_largura] = redimensionada
    return quadro


def esticar(imagem, lado):
    return cv2.resize(imagem, (lado, lado), interpolation=cv2.INTER_AREA)


def cortar_centro(imagem, lado):
    altura, largura = imagem.shape[:2]
    menor = min(altura, largura)
    topo, esquerda = (altura - menor) // 2, (largura - menor) // 2
    return cv2.resize(imagem[topo:topo + menor, esquerda:esquerda + menor], (lado, lado), interpolation=cv2.INTER_AREA)


for caminho in exemplos[:3]:
    cor = ler_cor(caminho)
    mostrar_linha([cor, esticar(cor, 128), cortar_centro(cor, 128), letterbox(cor, 128)],
                  ["original 360x480", "esticar 128", "cortar centro 128", "letterbox 128"])

### 4.2 Rotação
Estimativa do ângulo: detectar bordas (Canny), achar segmentos de reta longos (Hough probabilístico) e pegar a **mediana** dos ângulos dos segmentos mais horizontais que verticais.
É um heurístico frágil (foto sem linhas retas falha); serve pra medir *quanto* as fotos estão inclinadas antes de decidir se vale corrigir.

In [ ]:
def estimar_inclinacao(cinza):
    bordas = cv2.Canny(cv2.GaussianBlur(cinza, (5, 5), 0), 50, 150)
    segmentos = cv2.HoughLinesP(bordas, 1, np.pi / 180, threshold=60,
                                minLineLength=cinza.shape[1] // 4, maxLineGap=10)
    if segmentos is None:
        return None

    angulos = []
    for x1, y1, x2, y2 in segmentos.reshape(-1, 4):
        angulo = np.degrees(np.arctan2(y2 - y1, x2 - x1))
        if abs(angulo) < 45:
            angulos.append(angulo)
    return float(np.median(angulos)) if angulos else None


def girar(imagem, graus):
    altura, largura = imagem.shape[:2]
    matriz = cv2.getRotationMatrix2D((largura / 2, altura / 2), graus, 1.0)
    return cv2.warpAffine(imagem, matriz, (largura, altura), borderValue=0)


# Prova de conceito: giro uma foto por um ângulo conhecido e vejo se o estimador recupera.
base = ler_cinza(exemplos[0])
print("foto original estima:", estimar_inclinacao(base))
for graus in (-10, 5, 15):
    print(f"girada {graus:+d}° -> estima {estimar_inclinacao(girar(base, graus))}")

In [ ]:
# Distribuição do ângulo estimado numa amostra: as fotos de campo estão inclinadas?
amostra = [todas[i] for i in rng.choice(len(todas), 300, replace=False)]
angulos = [estimar_inclinacao(ler_cinza(c)) for c in amostra]
validos = [a for a in angulos if a is not None]

print(f"{len(validos)} de {len(angulos)} fotos com estimativa")
plt.hist(validos, bins=40)
plt.xlabel("ângulo estimado (graus)")
plt.title("inclinação estimada nas fotos de campo")
plt.show()

### 4.3 Augmentation (só no treino)
Variações aleatórias e **plausíveis em campo**: pequeno giro, brilho/contraste, leve desfoque, espelho horizontal **não** (dígitos espelhados não existem). Cada função recebe um `rng` pra ser reprodutível.
Vai ser aplicada só no split de treino (notebook 03); validação e teste ficam intactos.

In [ ]:
def aug_brilho_contraste(cor, rng, alfa=(0.7, 1.3), beta=(-30, 30)):
    # saída = alfa * entrada + beta  (alfa = contraste, beta = brilho)
    return cv2.convertScaleAbs(cor, alpha=rng.uniform(*alfa), beta=rng.uniform(*beta))


def aug_giro(cor, rng, max_graus=10):
    return girar(cor, rng.uniform(-max_graus, max_graus))


def aug_desfoque(cor, rng, chance=0.3):
    return cv2.GaussianBlur(cor, (5, 5), 0) if rng.random() < chance else cor


def augmentar(cor, rng):
    return aug_desfoque(aug_giro(aug_brilho_contraste(cor, rng), rng), rng)


rng_aug = np.random.default_rng(0)
for caminho in exemplos[:3]:
    cor = ler_cor(caminho)
    mostrar_linha([cor] + [augmentar(cor, rng_aug) for _ in range(5)],
                  ["original"] + [f"aug {i + 1}" for i in range(5)])

## 5. O que entra no pipeline da CNN

Decisões provisórias (revisar olhando as imagens acima):

1. **Tamanho:** letterbox quadrado (preserva a foto inteira; esticar distorce, cortar perde borda).
2. **Iluminação:** CLAHE como opção de pré-processamento; comparar com/sem na validação em vez de assumir.
3. **Nitidez:** Laplaciano fica como **feature de triagem** (heurística), não como entrada da CNN.
4. **Rotação:** estimador Hough é frágil; **não** corrigir na entrada. Giro pequeno vira augmentation.
5. **Kernels manuais:** didáticos; a CNN aprende os próprios. Servem de referência pra inspecionar os filtros da 1ª camada depois.
6. **Augmentation:** só no treino (4.3).